# Political Misinformation — Network Diffusion Analysis (UPFD / PolitiFact)

This notebook picks up where the local NLP + cascade-size analysis left off.
It uses the **UPFD** pre-built propagation graphs (Dou et al., 2021), which are
already-hydrated retweet trees for the same PolitiFact fake/real stories — no
Twitter API keys needed.

**What this notebook does:**
1. Installs PyTorch Geometric
2. Downloads the UPFD PolitiFact graphs
3. Computes real structural diffusion metrics (breadth, depth, density) per story
4. Statistically compares fake vs. real (mirrors the cascade-size test done locally)
5. Trains a simple GNN to classify fake vs. real from propagation structure alone
6. Saves plots + a results CSV you can merge with the local results

Run cells top to bottom. GPU runtime recommended (Runtime > Change runtime type > GPU).


In [ ]:
# 1. Install PyTorch Geometric (matched to Colab's pre-installed torch)
import torch
print("Torch version:", torch.__version__, "| CUDA:", torch.cuda.is_available())

!pip install torch_geometric -q


In [ ]:
# 2. Download the UPFD PolitiFact dataset (content features = title/text embeddings)
# This downloads from Google Drive automatically via PyG's dataset loader.
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader

train_dataset = UPFD(root='upfd_data', name='politifact', feature='content', split='train')
test_dataset  = UPFD(root='upfd_data', name='politifact', feature='content', split='test')

print("Train graphs:", len(train_dataset))
print("Test graphs:", len(test_dataset))
print("Example graph:", train_dataset[0])


In [ ]:
# 3. Combine splits for full-dataset structural analysis (we'll re-split later for the GNN)
all_graphs = list(train_dataset) + list(test_dataset)
print("Total graphs:", len(all_graphs))

# UPFD label convention: 1 = fake, 0 = real (root node is index 0 in each graph)
labels = [g.y.item() for g in all_graphs]
print("Fake:", sum(labels), "| Real:", len(labels) - sum(labels))


In [ ]:
# 4. Compute structural diffusion metrics per graph: breadth, depth, density
import networkx as nx
from torch_geometric.utils import to_networkx
import pandas as pd

records = []
for g in all_graphs:
    nxg = to_networkx(g, to_undirected=True)
    n_nodes = nxg.number_of_nodes()
    n_edges = nxg.number_of_edges()
    # root node = 0 (the news item itself, per UPFD convention)
    try:
        depths = nx.single_source_shortest_path_length(nxg, 0)
        max_depth = max(depths.values()) if depths else 0
    except Exception:
        max_depth = None
    density = nx.density(nxg) if n_nodes > 1 else 0
    records.append({
        "label": int(g.y.item()),  # 1=fake, 0=real
        "breadth_num_users": n_nodes - 1,  # exclude root news node
        "num_edges": n_edges,
        "max_depth": max_depth,
        "density": density
    })

struct_df = pd.DataFrame(records)
struct_df['label_name'] = struct_df['label'].map({1: 'fake', 0: 'real'})
struct_df.to_csv('upfd_structural_metrics.csv', index=False)
struct_df.groupby('label_name')[['breadth_num_users','max_depth','density']].describe()


In [ ]:
# 5. Statistical comparison: fake vs real (mirrors the local cascade-size Mann-Whitney test)
from scipy import stats

for metric in ['breadth_num_users', 'max_depth', 'density']:
    fake_vals = struct_df[struct_df['label']==1][metric].dropna()
    real_vals = struct_df[struct_df['label']==0][metric].dropna()
    u_stat, p_val = stats.mannwhitneyu(fake_vals, real_vals, alternative='two-sided')
    print(f"--- {metric} ---")
    print(f"Fake median: {fake_vals.median():.2f} | Real median: {real_vals.median():.2f}")
    print(f"Mann-Whitney U={u_stat:.1f}, p={p_val:.6f}, significant={p_val < 0.05}")
    print()


In [ ]:
# 6. Visualize the distributions
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['breadth_num_users', 'max_depth', 'density']):
    sns.boxplot(data=struct_df, x='label_name', y=metric, ax=ax)
    ax.set_title(metric)
    ax.set_yscale('log') if metric != 'density' else None
plt.tight_layout()
plt.savefig('upfd_structural_comparison.png', dpi=150)
plt.show()


In [ ]:
# 7. (Bonus) Train a simple GNN to classify fake vs real from propagation structure alone
# This tests: does WHO retweets and HOW the cascade branches, by itself, predict veracity?
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch.utils.data import random_split

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(in_channels=train_dataset.num_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

def train_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)
    return correct / total

for epoch in range(1, 31):
    loss = train_epoch()
    if epoch % 5 == 0:
        train_acc = evaluate(train_loader)
        test_acc = evaluate(test_loader)
        print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Train Acc {train_acc:.3f} | Test Acc {test_acc:.3f}")


In [ ]:
# 8. Save final structural results for merging with your local NLP + cascade-size results
struct_df.to_csv('upfd_structural_metrics.csv', index=False)
print("Saved upfd_structural_metrics.csv — download this and bring it back into your local analysis/writeup.")

from google.colab import files
files.download('upfd_structural_metrics.csv')
files.download('upfd_structural_comparison.png')


## Interpreting your results for the paper

- **Breadth (num_users)**: how many distinct users retweeted the story — direct reach.
- **Max depth**: how many "hops" the story traveled beyond direct retweets of the source — a proxy for cascade depth in Vosoughi et al.'s sense.
- **Density**: how interconnected the retweeting users were — a rough echo-chamber signal (higher density can indicate tighter, more clustered spread).
- **GNN accuracy**: if the GNN classifies fake vs. real well *using only graph structure* (no text), that's strong evidence that *how* something spreads is itself diagnostic — a nice complement to your text-based classifier's results.

Bring `upfd_structural_metrics.csv` back to merge with `processed_politifact.csv` (via the `id`/story identifiers) for a combined discussion section.
